In [ ]:
import pandas as pd
import networkx as nx
from CTDNE.ctdne import CTDNE


df = pd.read_csv("syn_data/syn_net_p0.8_mu0.2_1.csv")

df = df.rename(columns={
    "u": "source",
    "v": "destination",
    "timestamp": "time"
})


df = df.sort_values("time").reset_index(drop=True)
G = nx.MultiGraph()

for row in df.itertuples(index=False):
    u = row.source
    v = row.destination
    t = row.time

    G.add_edge(u, v, time=float(t))

CTDNE_model = CTDNE(
    G,
    dimensions=64,
    walk_length=30,
    num_walks=200,
    workers=4
)

model = CTDNE_model.fit(
    window=10,
    min_count=1,
    batch_words=4
)

model.wv.save_word2vec_format("node_embedding.emb")
#model.save("model_name.md")

Generating walks (CPU: 4): 100%|██████████| 50/50 [00:00<00:00, 198.95it/s]


[('38', 0.9838882088661194), ('18', 0.9829115271568298), ('43', 0.978462815284729), ('4', 0.9773052334785461), ('35', 0.8563947081565857), ('45', 0.8004758954048157), ('48', 0.7671477198600769), ('46', 0.7188845872879028), ('25', 0.6681907773017883), ('29', 0.6361683011054993)]


In [ ]:
import os
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd


def load_word2vec_emb(emb_path: str):
    emb_dict = {}

    with open(emb_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    if len(lines) == 0:
        raise ValueError(f"Empty embedding file: {emb_path}")

    first = lines[0].strip().split()

    has_header = False
    if len(first) == 2:
        try:
            int(first[0])
            int(first[1])
            has_header = True
        except ValueError:
            has_header = False

    data_lines = lines[1:] if has_header else lines

    emb_dim = None
    for line in data_lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        node = parts[0]
        vec = np.asarray([float(x) for x in parts[1:]], dtype=np.float32)
        if emb_dim is None:
            emb_dim = len(vec)
        elif len(vec) != emb_dim:
            raise ValueError(f"Inconsistent embedding dimension in line: {line[:100]}")
        emb_dict[node] = vec

    if emb_dim is None:
        raise ValueError(f"No valid embeddings found in {emb_path}")

    return emb_dict, emb_dim


def fixed_time_encoding_batch(z_values: np.ndarray, d_model: int):
    """
    对一批标准化后的时间差 z 做固定 sin-cos 编码

    参数:
        z_values: [N]
        d_model: 时间编码维度，必须是偶数

    返回:
        pe: [N, d_model]
    """
    if d_model % 2 != 0:
        raise ValueError("d_model must be even")

    z_values = np.asarray(z_values, dtype=np.float32)
    pe = np.zeros((len(z_values), d_model), dtype=np.float32)

    half = d_model // 2
    i = np.arange(half, dtype=np.float32)
    denom = np.power(10000.0, 2 * i / d_model).astype(np.float32)  # [half]

    pe[:, 0::2] = np.sin(z_values[:, None] / denom[None, :])
    pe[:, 1::2] = np.cos(z_values[:, None] / denom[None, :])

    return pe


def build_node_event_features(
    interaction_path: str,
    emb_path: str,
    out_dir: str,
    sep: str = ",",
    has_header: bool = True,
    source_col: str = "source",
    destination_col: str = "destination",
    time_col: str = "timestamp",
    time_dim: int = 32,
    sort_by_time: bool = True,
    fill_missing: str = "zero", 
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if has_header:
        df = pd.read_csv(interaction_path, sep=sep)
    else:
        df = pd.read_csv(
            interaction_path,
            sep=sep,
            header=None,
            names=[source_col, destination_col, time_col]
        )

    required_cols = {source_col, destination_col, time_col}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    df = df[[source_col, destination_col, time_col]].copy()
    df[time_col] = pd.to_numeric(df[time_col], errors="raise")

    if sort_by_time:
        df = df.sort_values(time_col).reset_index(drop=True)


    df["source_str"] = df[source_col].astype(str)
    df["destination_str"] = df[destination_col].astype(str)

    emb_dict, emb_dim = load_word2vec_emb(emb_path)

    # -------------------------
    # 3) 计算每个 node-event 的 delta time
    #    delta = 当前时间 - 上次该节点参与交互的时间
    #    首次出现节点，delta 设为 0
    # -------------------------
    last_time = {}
    src_delta = np.zeros(len(df), dtype=np.float32)
    dst_delta = np.zeros(len(df), dtype=np.float32)

    timestamps = df[time_col].to_numpy()
    src_nodes = df["source_str"].to_numpy()
    dst_nodes = df["destination_str"].to_numpy()

    for i in range(len(df)):
        t = float(timestamps[i])
        s = src_nodes[i]
        d = dst_nodes[i]

        if s in last_time:
            src_delta[i] = t - float(last_time[s])
        else:
            src_delta[i] = 0.0

        if d in last_time:
            dst_delta[i] = t - float(last_time[d])
        else:
            dst_delta[i] = 0.0

        last_time[s] = t
        last_time[d] = t


    all_delta = np.concatenate([src_delta, dst_delta], axis=0)
    time_mean = float(all_delta.mean())
    time_std = float(all_delta.std())

    if time_std < 1e-12:
        time_std = 1.0

    src_delta_z = (src_delta - time_mean) / time_std
    dst_delta_z = (dst_delta - time_mean) / time_std


    src_time_feat = fixed_time_encoding_batch(src_delta_z, d_model=time_dim)
    dst_time_feat = fixed_time_encoding_batch(dst_delta_z, d_model=time_dim)


    zero_emb = np.zeros(emb_dim, dtype=np.float32)

    src_node_feat = np.zeros((len(df), emb_dim), dtype=np.float32)
    dst_node_feat = np.zeros((len(df), emb_dim), dtype=np.float32)

    missing_nodes = set()

    for i in range(len(df)):
        s = src_nodes[i]
        d = dst_nodes[i]

        if s in emb_dict:
            src_node_feat[i] = emb_dict[s]
        else:
            missing_nodes.add(s)
            if fill_missing == "zero":
                src_node_feat[i] = zero_emb
            else:
                raise KeyError(f"Node {s} not found in embedding file.")

        if d in emb_dict:
            dst_node_feat[i] = emb_dict[d]
        else:
            missing_nodes.add(d)
            if fill_missing == "zero":
                dst_node_feat[i] = zero_emb
            else:
                raise KeyError(f"Node {d} not found in embedding file.")

    src_feat = np.concatenate([src_node_feat, src_time_feat], axis=1).astype(np.float32)
    dst_feat = np.concatenate([dst_node_feat, dst_time_feat], axis=1).astype(np.float32)


    meta_df = pd.DataFrame({
        "event_id": np.arange(len(df), dtype=np.int64),
        "source": df[source_col].to_numpy(),
        "destination": df[destination_col].to_numpy(),
        "timestamp": timestamps,
        "src_delta": src_delta,
        "dst_delta": dst_delta,
        "src_delta_z": src_delta_z,
        "dst_delta_z": dst_delta_z,
    })

    meta_df.to_csv(out_dir / "metadata.csv", index=False)

    np.save(out_dir / "src_feat.npy", src_feat)
    np.save(out_dir / "dst_feat.npy", dst_feat)


    config = {
        "interaction_path": str(interaction_path),
        "emb_path": str(emb_path),
        "num_events": int(len(df)),
        "embedding_dim": int(emb_dim),
        "time_dim": int(time_dim),
        "output_dim": int(emb_dim + time_dim),
        "time_mean": time_mean,
        "time_std": time_std,
        "sort_by_time": bool(sort_by_time),
        "fill_missing": fill_missing,
        "num_missing_nodes": int(len(missing_nodes)),
        "missing_nodes_sample": sorted(list(missing_nodes))[:50],
    }

    with open(out_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    print(f"Done. Output saved to: {out_dir}")
    print(f"num_events = {len(df)}")
    print(f"embedding_dim = {emb_dim}")
    print(f"time_dim = {time_dim}")
    print(f"output_dim = {emb_dim + time_dim}")
    print(f"time_mean = {time_mean:.6f}")
    print(f"time_std = {time_std:.6f}")
    print(f"missing_nodes = {len(missing_nodes)}")


if __name__ == "__main__":
    # ===== 你只需要改这里 =====
    FILE_NAME = "p0.8_mu0.2_1"
    interaction_path = f"../syn_data/{FILE_NAME}.csv"
    emb_path = f"{FILE_NAME}.emb" 

    build_node_event_features(
        interaction_path=interaction_path,
        emb_path=emb_path,
        out_dir=FILE_NAME,
        sep=",",
        has_header=True,
        source_col="source",
        destination_col="destination",
        time_col="timestamp",
        time_dim=32,
        sort_by_time=True,
        fill_missing="zero"
    )

Done. Output saved to: node_event_feat.emb
num_events = 469
embedding_dim = 64
time_dim = 32
output_dim = 96
time_mean = 133.761200
time_std = 140.985092
missing_nodes = 0
